<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 13 · Logistic Regression:<br>Learning to Classify</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 28 November 2026 &nbsp; | &nbsp; Week 7 · Part 3: Supervised Learning</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> In L11–L12, we predicted numerical values such as house prices. What if the answer must be a <em>category</em>? How can a model learn class probabilities and decide which label to predict?</div>
</div>

### What you will be able to do

By the end of this lecture, you should be able to:

- Distinguish a numerical regression target from a categorical classification target.
- Explain why unrestricted linear predictions cannot simply be interpreted as probabilities.
- Calculate a **linear score**, apply the **sigmoid**, and interpret the probability of a clearly specified positive class.
- Convert binary probabilities into decisions and derive a **linear decision boundary** from a score equation.
- Interpret the signs of logistic-regression coefficients and the meaning of **log-odds** at an introductory level.
- Explain why **binary logarithmic loss** penalizes confident wrong predictions.
- Train a leakage-safe `Pipeline`, interpret `classes_`, `predict_proba()`, `predict()`, and connect fitted coefficients to predictions.
- Extend the idea to **multinomial logistic regression**, calculate a **softmax** example, and inspect three-class probabilities.


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">From house prices to categories</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">One dataset can support different questions. The prediction question determines the target.</div>
</div>

### Start exactly where L12 stopped

In L11, we learned to predict the **sale price** of a house. In L12, we studied whether those numerical predictions generalize to unseen houses. A familiar regression model is

$$\hat y=b_0+b_1x_1+\cdots+b_px_p.$$

Its output is a number, such as an estimated sale price of **USD 185,000**. Now change only the question:

> **Will this house sell for at least USD 200,000?**

The cutoff of **USD 200,000 is illustrative**. The target now has two possible labels:

$$y=\begin{cases}1 & \text{sale price }\geq\$200{,}000,\\0 & \text{sale price }<\$200{,}000.\end{cases}$$

| Illustrative house | Original sale price (continuous target) | New binary target |
|:--|--:|--:|
| A | USD 185,000 | 0 |
| B | USD 242,000 | 1 |
| C | USD 199,500 | 0 |
| D | USD 215,000 | 1 |

The houses and their measurements need not change. **Our desired answer has changed.**

**Think:** Is the new task regression or classification? What would the model have to output?

**Interpretation.** This is **binary classification**: each observation belongs to one of two categories. The numbers 0 and 1 are *codes for labels*, not prices. We **could** predict price with a regression model and then apply the USD 200,000 cutoff; that is a legitimate indirect baseline. However, a **direct classifier** models the category (and often an estimated class probability) explicitly. If we still care about the exact price, regression remains appropriate.

We now leave Ames Housing behind. Our real classification task returns to the Palmer Penguins dataset.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">Define the Penguins prediction task</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Keep the original species target; then construct one explicit binary question.</div>
</div>

### A familiar scientific dataset, now with labels to predict

The Palmer Penguins CSV contains **344 observations** and **eight attributes**. Species is a **nominal categorical target** with three values: **Adelie**, **Chinstrap**, and **Gentoo**. Its labels are not amounts: there is no meaningful distance between species names.

For our first classifier, ask:

> **Is this penguin an Adelie penguin?**

We deliberately define **1 = Adelie** (the *positive class*) and **0 = Not Adelie** (Chinstrap **or** Gentoo). We are **not** discarding the original `species` labels; they remain available for the multiclass extension.

The model initially receives only the two physical measurements `bill_length_mm` and `bill_depth_mm` as features. The target column and the split label must never be included among the inputs.

In [ ]:
# One setup cell for all examples in L13; no downloads or previous notebooks needed.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

NAVY, TEAL, GOLD, BLUE = "#102A43", "#18B7AC", "#DB9137", "#557AB5"
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": GOLD, "Gentoo": BLUE}
BINARY_COLORS = {0: GOLD, 1: "#158B94"}
MEASUREMENTS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
BINARY_FEATURES = ["bill_length_mm", "bill_depth_mm"]
plt.rcParams.update({"figure.dpi": 115, "font.size": 10.5,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.labelcolor": NAVY,
                     "text.color": NAVY, "axes.titlecolor": NAVY})

DATA_PATH = Path("data/penguins.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError("Expected data/penguins.csv next to the notebook. Extract the entire L13 ZIP first.")
penguins = pd.read_csv(DATA_PATH)
assert penguins.shape == (344, 8), "Unexpected source dataset. Check the packaged CSV."
print("Original dataset:", penguins.shape)
display(penguins.head(4))

**Interpretation.** The dataset includes the species labels that a classifier can learn from. For an unseen observation, we will supply its bill measurements, **not** its species.

The original file has **two rows** with all four physical measurements missing. As in L08–L10, we use a complete-measurement classroom subset of **342 rows**.

In [ ]:
# EXACT classroom split recipe from L08–L10, applied to the ORIGINAL row indices.
study = penguins.dropna(subset=MEASUREMENTS).copy()
rng = np.random.default_rng(8)
row_order = rng.permutation(study.index.to_numpy())
n_train = int(0.70 * len(study))
n_val = int(0.20 * len(study))
study["split_role"] = "test"
study.loc[row_order[:n_train], "split_role"] = "train"
study.loc[row_order[n_train:n_train + n_val], "split_role"] = "validation"

# Species is retained; the new binary target is an ADDITIONAL column.
study["is_adelie"] = study["species"].eq("Adelie").astype(int)
train = study.loc[study["split_role"] == "train"].copy()
validation = study.loc[study["split_role"] == "validation"].copy()
test = study.loc[study["split_role"] == "test"].copy()  # Protected; no predictions in L13.

assert len(study) == 342
assert (len(train), len(validation), len(test)) == (239, 68, 35)
assert not (set(train.index) & set(validation.index) or
            set(train.index) & set(test.index) or set(validation.index) & set(test.index))
assert set(study["is_adelie"].unique()) == {0, 1}

print("Classroom partition × original species:")
display(pd.crosstab(study["split_role"], study["species"])
        .reindex(index=["train", "validation", "test"], columns=SPECIES_ORDER))
print("Binary target counts in training: 1 = Adelie, 0 = Not Adelie")
display(train["is_adelie"].value_counts().sort_index().rename_axis("class"))

**Interpretation.** The split contains **239 training**, **68 validation**, and **35 test** observations; all three original species appear in each partition. The random generator and original row indices reproduce L08–L10's classroom illustration. 

The training observations are used to learn preprocessing parameters and model coefficients. We will show a few validation examples to understand prediction outputs; the **test set stays untouched** for later evaluation work.

In [ ]:
# Draw the training observations in the two-feature plane.
fig, ax = plt.subplots(figsize=(8.8, 5.0))
for cls, label in [(1, "Adelie (1)"), (0, "Not Adelie (0)")]:
    points = train.loc[train["is_adelie"] == cls]
    ax.scatter(points["bill_length_mm"], points["bill_depth_mm"],
               s=40, alpha=.75, color=BINARY_COLORS[cls],
               edgecolor="white", linewidth=.45, label=label)
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)",
       title="Our first classification question: Adelie or not?")
ax.legend(frameon=False)
ax.grid(alpha=.15)
plt.show()

**Interpretation.** We see two labeled groups in a **two-dimensional feature space**, a representation familiar from L09. Their points overlap in some regions, so a simple straight boundary may not perfectly separate every observation. Our challenge is to learn a decision from examples rather than draw a boundary by hand.


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">Why not use an unrestricted line as a probability?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Regression can create labels after thresholding, but its outputs are not automatically probabilities.</div>
</div>

### Reuse the L11 linear model—now with 0/1 labels

A straight-line regression model trained on binary labels still computes

$$\hat y=b_0+b_1x.$$

**Nothing in this equation restricts** $\hat y$ to the interval $[0,1]$. A prediction of **1.3** might be a number, but it cannot directly be a **130% probability**. Likewise, **−0.2** cannot be interpreted as a negative probability.

Regression followed by a threshold can sometimes be a useful simple classifier; what is missing is a well-defined probability model specifically designed for a binary response. Squared-error fitting is not the natural objective when the target indicates an event.

In [ ]:
# Use the ALREADY-STUDIED LinearRegression as a diagnostic, not as our final classifier.
line_baseline = LinearRegression().fit(train[["bill_length_mm"]], train["is_adelie"])
line_val = line_baseline.predict(validation[["bill_length_mm"]])
print(f"Fitted line: y_hat = {line_baseline.intercept_:.3f} "
      f"+ ({line_baseline.coef_[0]:.4f}) × bill_length_mm")
print(f"Validation output range: {line_val.min():.3f} to {line_val.max():.3f}")
print("Outputs below 0:", int((line_val < 0).sum()),
      "| Outputs above 1:", int((line_val > 1).sum()))
assert ((line_val < 0) | (line_val > 1)).any()

In [ ]:
# Same real training labels, two DIFFERENT illustrations:
# left: the fitted regression line; right: an S-shaped probability idea (NOT a fitted classifier).
length_grid = np.linspace(32, 60, 250)
line_grid = line_baseline.predict(pd.DataFrame({"bill_length_mm": length_grid}))
illustrative_s_curve = 1 / (1 + np.exp(0.45 * (length_grid - 44.5)))
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.7), sharex=True)
for ax in axes:
    for cls in [0, 1]:
        sub = train.loc[train["is_adelie"] == cls]
        ax.scatter(sub["bill_length_mm"], sub["is_adelie"], s=16,
                   alpha=.24, color=BINARY_COLORS[cls])
    ax.set(xlabel="Bill length (mm)", ylabel="Observed class (0 or 1)")
    ax.axhline(0, color=NAVY, lw=.7, alpha=.25)
    ax.axhline(1, color=NAVY, lw=.7, alpha=.25)
    ax.grid(alpha=.12)
axes[0].plot(length_grid, line_grid, color=NAVY, lw=2.5, label="Fitted least-squares line")
axes[0].set(title="Unrestricted numerical predictions", ylim=(-.85, 1.55))
axes[1].plot(length_grid, illustrative_s_curve, color=TEAL, lw=2.7,
             label="Illustrative S-curve (not fitted)")
axes[1].set(title="A bounded function could represent probabilities", ylim=(-.09, 1.09))
for ax in axes: ax.legend(loc="upper right", frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**Interpretation.** The *fitted* line produces values outside $[0,1]$; the *illustrative* S-curve remains between zero and one. **The right-hand curve has not been learned from the data**—it previews the kind of transformation we will explain next. Both panels show the same observed training labels, but the vertical scales differ intentionally to reveal the line's invalid probability range.

This motivates our next question:

> **Could we keep the useful linear combination of features, but transform its score into a valid probability?**

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">Linear scores and the sigmoid function</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Retain the familiar weighted sum; change what we do with its output.</div>
</div>

### Step 1 — Build a linear score (not a probability)

For one penguin represented by $p$ features, write

$$z=b_0+b_1x_1+b_2x_2+\cdots+b_px_p.$$

- $x_j$: the penguin's value for feature $j$.
- $b_j$: a coefficient learned from training observations.
- $b_0$: the intercept, a score offset.
- $z$: a **linear score** that may be any real number.

This is the same weighted-sum structure you know from L11. The major change is that **we do not call $z$ the predicted class or probability**.

### Step 2 — Transform the score with a sigmoid

The **sigmoid** (also called the *logistic function*) is

$$\boxed{\sigma(z)=\frac{1}{1+e^{-z}}},\qquad e\approx2.71828.$$

The constant $e$ is the base of the natural logarithm. For our binary Penguins question, define

$$\boxed{P(Y=1\mid X=x)=\sigma(z)}.$$

This means: *given this penguin's feature values $x$, the fitted model estimates the probability that the penguin is Adelie (the class coded 1).* We obtain $P(Y=0\mid X=x)=1-\sigma(z)$.

In [ ]:
def sigmoid(z):
    """Convert a (small or moderate) numerical score to a binary probability."""
    z = np.asarray(z, dtype=float)
    return 1 / (1 + np.exp(-z))

scores = np.array([-2, -1, 0, 1, 2], dtype=float)
sigmoid_table = pd.DataFrame({"linear score z": scores.astype(int),
                              "sigmoid(z)": np.round(sigmoid(scores), 3)})
display(sigmoid_table)
assert np.allclose(sigmoid(scores), [0.1192029, 0.2689414, 0.5, 0.7310586, 0.8807971], atol=1e-7)

z_grid = np.linspace(-6, 6, 500)
fig, ax = plt.subplots(figsize=(9.2, 4.8))
ax.plot(z_grid, sigmoid(z_grid), color=TEAL, lw=3, label=r"$\sigma(z)=1/(1+e^{-z})$")
ax.axhline(.5, color=GOLD, ls="--", lw=1.3)
ax.axvline(0, color=GOLD, ls="--", lw=1.3)
ax.scatter(scores, sigmoid(scores), color=NAVY, s=44, zorder=4)
for zz in [-2, 0, 2]:
    ax.annotate(f"({zz}, {sigmoid(zz):.3f})", (zz, float(sigmoid(zz))),
                xytext=(8, 12 if zz != 2 else -19), textcoords="offset points", fontsize=9)
ax.set(xlabel="Linear score (z)", ylabel="Predicted probability P(Y = 1 | X)",
       title="The sigmoid turns an unrestricted score into a probability",
       xlim=(-6.2, 6.2), ylim=(-.04, 1.04))
ax.grid(alpha=.15)
ax.legend(frameon=False, loc="upper left")
plt.show()

**Interpretation.** Large negative scores produce probabilities close to 0, a score of **0 gives exactly 0.5**, and large positive scores produce probabilities close to 1. The output is strictly between 0 and 1 for every finite score. Notice that moving the score by one unit does **not** change probability by a fixed amount: the curve flattens at its ends.


### Worked prediction — calculate it before calling a library

For a *constructed* Adelie model, suppose bill measurements have already been standardized and

$$z=-1-x_1+2x_2.$$

Here $x_1$ is standardized bill length and $x_2$ is standardized bill depth. They are **not** raw millimeters. Consider an illustrative penguin with $x_1=0$ and $x_2=1$:

$$z=-1-(0)+2(1)=1.$$

Now apply the sigmoid:

$$P(\text{Adelie}\mid x)=\frac{1}{1+e^{-1}}\approx0.731.$$

The model would assign approximately **73.1% to Adelie** and **26.9% to Not Adelie**. We have not yet made a class decision; that is a separate step.

In [ ]:
# Verify the hand calculation; coefficients above are ILLUSTRATIVE, not fitted.
x1_example, x2_example = 0.0, 1.0
z_example = -1 - x1_example + 2 * x2_example
p_example = float(sigmoid(z_example))
print(f"Linear score: {z_example:.1f}")
print(f"P(Adelie): {p_example:.3f}")
print(f"P(Not Adelie): {1 - p_example:.3f}")
assert np.isclose(z_example, 1.0) and np.isclose(p_example, 0.7310586, atol=1e-7)

**Interpretation.** The coefficients determine the **score**. The sigmoid converts that score into a **binary class probability**. A predicted probability reflects what this model learned from its available features and examples.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">From probability to class and boundary</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">A probability estimate and a decision are related, but they are not the same object.</div>
</div>

### When does the model predict class 1?

For our educational binary decision rule, use threshold $t=0.5$:

$$
\hat y=\begin{cases}
1 & \text{if } p\geq0.5,\\
0 & \text{if } p<0.5.
\end{cases}
$$

This is an explicitly specified **decision rule**, not a mathematical requirement that 0.5 is always the best choice. Threshold selection, its consequences, and responsible evaluation will be studied in L14.

| Predicted probability of Adelie | Predicted class at 0.5 |
|--:|:--|
| 0.20 | Not Adelie (0) |
| 0.49 | Not Adelie (0) |
| 0.50 | Adelie (1) |
| 0.72 | Adelie (1) |
| 0.95 | Adelie (1) |

**Technical nuance:** Exact 0.5 ties require a tie convention. This table defines $p\geq0.5$ as class 1. A library may use a different exact-tie convention; real-valued fitted predictions almost never land at an exact tie.

In [ ]:
toy_probabilities = np.array([0.20, 0.49, 0.50, 0.72, 0.95])
toy_classes = np.where(toy_probabilities >= .5, 1, 0)
display(pd.DataFrame({"P(Adelie)": toy_probabilities,
                      "Predicted binary label": toy_classes,
                      "Predicted class": np.where(toy_classes == 1, "Adelie", "Not Adelie")}))

### What do the coefficients actually mean? Introducing log-odds

Suppose $p=P(Y=1\mid X=x)$. The **odds** in favor of class 1 are

$$
\mathrm{odds}=\frac{p}{1-p}.
$$

Odds compare the probability of class 1 with the probability of class 0. Taking the **natural logarithm** gives the **log-odds** (or *logit*):

$$
\boxed{\log\left(\frac{p}{1-p}\right)=b_0+b_1x_1+\cdots+b_px_p=z}.
$$

This answers why the algorithm is named **logistic regression**: the model is **linear in the log-odds**, even though it is used for **classification** and is **not linear in probability**.

- A **positive** $b_j$ increases the score and therefore the probability of class 1 as feature $x_j$ increases, **holding the other features fixed**.
- A **negative** $b_j$ decreases that probability under the same condition.
- $b_j$ does **not** mean that probability increases by a fixed $b_j$ percentage points.


In [ ]:
# Same probabilities, now viewed as odds and log-odds.
z_values = np.array([-1., 0., 1.])
probabilities = sigmoid(z_values)
odds = probabilities / (1 - probabilities)
log_odds = np.log(odds)
display(pd.DataFrame({"score z": z_values.astype(int),
                      "probability p": np.round(probabilities, 3),
                      "odds p/(1-p)": np.round(odds, 3),
                      "log-odds": np.round(log_odds, 3)}))
assert np.allclose(log_odds, z_values)

**Interpretation.** At $z=0$, the odds are 1-to-1, corresponding to a probability of 0.5. At $z=1$, the odds are approximately $e:1$, and the probability is about 0.731. 

### A straight decision boundary from a nonlinear sigmoid

The sigmoid equals 0.5 **exactly when its input score is zero**:

$$
p=0.5 \iff z=0.
$$

For two features, the boundary equation is

$$
\boxed{b_0+b_1x_1+b_2x_2=0}.
$$

For our **constructed** score $z=-1-x_1+2x_2$, this becomes

$$
-1-x_1+2x_2=0 \quad\Longrightarrow\quad x_2=\frac{1+x_1}{2}.
$$

That is a **straight line** in the two-feature plane. The sigmoid curve is nonlinear in score space; the *decision boundary* is a straight line in feature space when the score uses the two features linearly.

In [ ]:
# Constructed geometry: show p=0.5 boundary and probability regions.
axis_values = np.linspace(-2.5, 2.5, 220)
x1_mesh, x2_mesh = np.meshgrid(axis_values, axis_values)
z_mesh = -1 - x1_mesh + 2 * x2_mesh
p_mesh = sigmoid(z_mesh)
fig, ax = plt.subplots(figsize=(7.6, 5.1))
cf = ax.contourf(x1_mesh, x2_mesh, p_mesh, levels=np.linspace(0, 1, 12),
                 cmap="BrBG", vmin=0, vmax=1, alpha=.78)
ax.contour(x1_mesh, x2_mesh, p_mesh, levels=[.5], colors=[NAVY], linewidths=2.2)
example_points = np.array([[-1.4, 1.0], [-.5, .8], [1.0, 1.7],
                           [-1.0, -.8], [.7, -.7], [1.7, .3]])
example_labels = (sigmoid(-1 - example_points[:, 0] + 2*example_points[:, 1]) >= .5).astype(int)
for cls, label in [(1, "Illustrative class 1"), (0, "Illustrative class 0")]:
    sub = example_points[example_labels == cls]
    ax.scatter(sub[:, 0], sub[:, 1], color=BINARY_COLORS[cls], s=95,
               edgecolors="white", linewidths=1.5, label=label, zorder=5)
ax.set(xlabel="Standardized feature x₁", ylabel="Standardized feature x₂",
       title="Constructed two-feature decision boundary", xlim=(-2.5, 2.5), ylim=(-2.5, 2.5))
ax.legend(loc="lower right", frameon=True, facecolor="white", framealpha=.9, fontsize=9)
cbar = fig.colorbar(cf, ax=ax, pad=.025)
cbar.set_label("Predicted probability P(Y = 1)")
plt.show()

**Interpretation.** The dark boundary marks $p=0.5$ or equivalently $z=0$. On one side, the model predicts class 1; on the other, class 0. The background changes smoothly because the model assigns **probabilities**, not only labels. 

**Think:** If you move across the 0.5 line, does the probability jump suddenly? 

No: the **predicted label** changes abruptly at the threshold, but the **probability** varies continuously.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">How does logistic regression learn?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Choose coefficients that assign large probabilities to observed classes.</div>
</div>

### The learning question changes with the target

In L11, linear regression used squared residuals. But now the target $y$ is a binary label and the prediction $p$ is a probability of class 1. A good objective should behave like this:

- If the true class is **1**, predicting $p=0.9$ should receive a **small penalty**, while $p=0.1$ should receive a **large penalty**.
- If the true class is **0**, predicting $p=0.1$ should receive a small penalty, while $p=0.9$ should receive a large penalty.
- Especially confident **wrong** probabilities should be penalized strongly.

This naturally motivates **binary logarithmic loss** (also called *binary log loss*).

In [ ]:
# Verify the proposed behavior before showing the general formula.
loss_examples = pd.DataFrame({"true class y": [1, 1, 0, 0],
                              "predicted p = P(Y=1)": [.9, .1, .1, .9]})
loss_examples["penalty"] = np.where(
    loss_examples["true class y"].eq(1),
    -np.log(loss_examples["predicted p = P(Y=1)"]),
    -np.log(1 - loss_examples["predicted p = P(Y=1)"])
)
display(loss_examples.round(3))
assert np.isclose(loss_examples.loc[0, "penalty"], -np.log(.9))
assert np.isclose(loss_examples.loc[1, "penalty"], -np.log(.1))

### The general formula, after the intuition

For **one binary-labeled observation**, define

$$
\boxed{L(y,p)=-\left[y\log(p)+(1-y)\log(1-p)\right]}.
$$

- $y$ is the **observed** class (0 or 1).
- $p$ is the model's **predicted probability of class 1**.
- $\log$ is the **natural logarithm**.
- $L$ is a **nonnegative penalty** to be minimized.

Because $y$ is either 0 or 1, one of the two terms disappears:

$$
\begin{aligned}
\text{If }y=1:\quad &L(1,p)=-\log(p),\\[4pt]
\text{If }y=0:\quad &L(0,p)=-\log(1-p).
\end{aligned}
$$

For example, if the observed label is Adelie and $p=0.1$, the penalty is $-\log(0.1)\approx2.303$. If $p=0.9$, the penalty is only about **0.105**. For a Not-Adelie label, reverse the interpretation of $p$.

In [ ]:
# Make the two loss cases visible. Avoid 0 and 1 exactly: log(0) diverges.
p_axis = np.linspace(.005, .995, 350)
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.plot(p_axis, -np.log(p_axis), color=TEAL, lw=2.7,
        label="Actual class 1:  −log(p)")
ax.plot(p_axis, -np.log(1 - p_axis), color=GOLD, lw=2.7,
        label="Actual class 0:  −log(1 − p)")
ax.scatter([.9, .1], [-np.log(.9), -np.log(.1)],
           color=TEAL, s=60, edgecolor="white", zorder=4)
ax.set(xlabel="Predicted probability p = P(Y = 1)", ylabel="Binary logarithmic loss",
       title="Confident mistakes receive a large penalty", ylim=(0, 4.6))
ax.grid(alpha=.15)
ax.legend(frameon=False)
plt.show()

### From an observation's loss to a learned model

A logistic-regression algorithm chooses its coefficients to make an **aggregate training loss** small (typically the mean or sum across training observations, with additional terms depending on the estimator). Equivalently, the probabilities should give the observed training labels a high *likelihood*. We **will not derive gradients, optimization updates, or formal maximum-likelihood estimation** today.

**Implementation note:** scikit-learn's `LogisticRegression` applies **regularization by default**, so its full objective adds a penalty on coefficients. Detailed optimization belongs to **L26**, when we study learning from errors.


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">Train logistic regression on real Penguins</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Use familiar pipelines; interpret what the new estimator actually learns and outputs.</div>
</div>

### A familiar preparation workflow, followed by a new model

We now replace the constructed coefficients with coefficients **learned from real observations**.

Our first model uses two input features so we can visualize its boundary. The familiar pipeline is:

1. `SimpleImputer(strategy="median")`: learn numerical filling values from **training data** (the working subset is complete on these measurements, so it has no values to fill here).
2. `StandardScaler()`: learn training means and standard deviations so features are on comparable numerical scales.
3. `LogisticRegression()`: learn coefficients for a binary class-probability model.

Scaling is particularly useful for a regularized logistic-regression estimator. The model receives only `X_train`; `species`, `is_adelie`, and `split_role` are never included in its features.

In [ ]:
# Separate features X from target y; fit only on TRAINING rows.
X_train = train[BINARY_FEATURES].copy()
y_train = train["is_adelie"].copy()
X_val = validation[BINARY_FEATURES].copy()
y_val = validation["is_adelie"].copy()

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])
binary_pipeline.fit(X_train, y_train)

binary_model = binary_pipeline.named_steps["classifier"]
print("Training features:", X_train.shape)
print("Estimated classes (in API order):", binary_model.classes_)
print("Learned intercept:", np.round(binary_model.intercept_, 3))
print("Learned standardized coefficients:", np.round(binary_model.coef_, 3))
assert binary_model.classes_.tolist() == [0, 1]

**Interpretation.** `fit(X_train, y_train)` learns **preprocessing parameters** and **classifier coefficients** using the 239 training observations only. The learned intercept and coefficients refer to **standardized** bill measurements after imputation. They do not mean “probability points per raw millimeter.”

The signs are instructive: a larger standardized bill length, holding depth fixed, tends to **lower** the model's Adelie probability; greater standardized bill depth, holding length fixed, tends to **raise** it. These are conditional model relationships, not causal statements.

In [ ]:
# Show what the classifier learned and connect it to z = b0 + b1*x1 + b2*x2.
learned_b0 = float(binary_model.intercept_[0])
learned_b = binary_model.coef_[0]
print("Fitted score in standardized coordinates:")
print(f"  z = {learned_b0:+.3f} {learned_b[0]:+.3f}*x1 {learned_b[1]:+.3f}*x2")
print("Feature order: x1 = standardized bill_length_mm, x2 = standardized bill_depth_mm")
print("Training scaler means:", np.round(binary_pipeline.named_steps["scaler"].mean_, 3))
print("Training scaler scales:", np.round(binary_pipeline.named_steps["scaler"].scale_, 3))

### `predict_proba()`, `predict()`, and `classes_`

These methods answer **different** questions:

- `predict_proba(X_new)` returns estimated probabilities for **each class**.
- `predict(X_new)` returns **one predicted class label**.
- `classes_` identifies the **order of probability columns**.

Never assume a probability column's meaning solely from its position. For this fitted binary model, `classes_` is `[0, 1]`: the first probability is **Not Adelie** and the second is **Adelie**. For other labels or different encodings, inspect the estimator instead of assuming.

In [ ]:
# Inspect a small set of previously unseen VALIDATION observations.
# These row identifiers are original CSV indices, retained from L08–L10.
example_ids = [63, 245, 335, 330, 129]
assert set(example_ids).issubset(X_val.index)
val_probabilities = binary_pipeline.predict_proba(X_val)
val_predictions = binary_pipeline.predict(X_val)
positive_col = int(np.flatnonzero(binary_model.classes_ == 1)[0])
negative_col = int(np.flatnonzero(binary_model.classes_ == 0)[0])

example_locations = X_val.index.get_indexer(example_ids)
examples = validation.loc[example_ids, ["species"] + BINARY_FEATURES].copy()
examples["P(Not Adelie)"] = val_probabilities[example_locations, negative_col]
examples["P(Adelie)"] = val_probabilities[example_locations, positive_col]
examples["Predicted class"] = np.where(val_predictions[example_locations] == 1,
                                       "Adelie", "Not Adelie")
display(examples.round(3))

assert np.all((val_probabilities >= 0) & (val_probabilities <= 1))
assert np.allclose(val_probabilities.sum(axis=1), 1.0)
assert np.all(np.isin(val_predictions, [0, 1]))

**Interpretation.** The classifier assigns probability **0.889** to Adelie for validation row 63 (an actual Adelie) and approximately **0.003** for row 245 (an actual Gentoo). Row 335 receives approximately **0.493**, close to the decision boundary, while row 330 is assigned approximately **0.567** and is predicted Adelie even though the original label is Chinstrap. Row 129, an actual Adelie, receives approximately **0.442** and is predicted Not Adelie.

These examples demonstrate how probabilities and decisions work; they are **not a complete model evaluation**. Disagreements motivate L14, where we will summarize and interpret such outcomes systematically.

In [ ]:
# Verify the connection between fitted score, sigmoid, probability and predicted label.
validation_scores = binary_pipeline.decision_function(X_val)
manual_positive_probs = sigmoid(validation_scores)
assert np.allclose(manual_positive_probs, val_probabilities[:, positive_col], atol=1e-12)
assert np.all(val_predictions == np.where(validation_scores > 0, 1, 0))
assert not np.any(np.isclose(validation_scores, 0.0, atol=1e-12))

print("First five VALIDATION examples: score → sigmoid(score) → predicted class")
display(pd.DataFrame({"score z": validation_scores[:5],
                      "sigmoid(z)": manual_positive_probs[:5],
                      "predict()": val_predictions[:5]},
                     index=X_val.index[:5]).round(3))
print("Check: sigmoid(decision_function) equals P(Adelie) from predict_proba().")

**Interpretation.** The numerical score from the fitted model **is exactly what the sigmoid transforms** to obtain the class-1 probability. A positive score corresponds to an Adelie probability above 0.5; a negative score corresponds to a probability below 0.5. In the absence of exact ties, scikit-learn's predicted labels agree with this decision rule.

Next, we inspect the geometry of the *same fitted model*.

In [ ]:
# Visualize the LEARNED two-dimensional boundary and probability field.
# Draw on the original millimeter axes; the pipeline handles standardization internally.
x_min, x_max = study["bill_length_mm"].min() - 2, study["bill_length_mm"].max() + 2
y_min, y_max = study["bill_depth_mm"].min() - 1, study["bill_depth_mm"].max() + 1
length_values = np.linspace(x_min, x_max, 220)
depth_values = np.linspace(y_min, y_max, 190)
xx, yy = np.meshgrid(length_values, depth_values)
mesh_df = pd.DataFrame({"bill_length_mm": xx.ravel(), "bill_depth_mm": yy.ravel()})
probability_surface = binary_pipeline.predict_proba(mesh_df)[:, positive_col].reshape(xx.shape)
score_surface = binary_pipeline.decision_function(mesh_df).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.1), sharex=True, sharey=True)
for ax in axes:
    for cls, label in [(1, "Adelie (train)"), (0, "Not Adelie (train)")]:
        sub = train.loc[train["is_adelie"] == cls]
        ax.scatter(sub["bill_length_mm"], sub["bill_depth_mm"],
                   color=BINARY_COLORS[cls], label=label, s=30,
                   alpha=.58, edgecolor="white", linewidth=.38, zorder=3)
    ax.contour(xx, yy, score_surface, levels=[0], colors=[NAVY], linewidths=2.2, zorder=5)
    ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)",
           xlim=(x_min, x_max), ylim=(y_min, y_max))
    ax.grid(alpha=.10)

axes[0].set_title("A. Training labels and learned boundary")
axes[0].legend(loc="upper right", fontsize=8.4, frameon=True, facecolor="white")
mapplot = axes[1].contourf(xx, yy, probability_surface, levels=np.linspace(0, 1, 16),
                           cmap="BrBG", vmin=0, vmax=1, alpha=.58, zorder=0)
axes[1].set_title("B. Model's predicted P(Adelie)")
for row_id in [63, 245, 335]:
    r = validation.loc[row_id]
    axes[1].scatter(r["bill_length_mm"], r["bill_depth_mm"],
                    marker="*", s=150, facecolors="white", edgecolors=NAVY,
                    linewidths=1.1, zorder=7)
    axes[1].annotate(str(row_id), (r["bill_length_mm"], r["bill_depth_mm"]),
                     xytext=(6, 8), textcoords="offset points", fontsize=8.5,
                     color=NAVY, fontweight="bold")
fig.colorbar(mapplot, ax=axes[1], fraction=.046, pad=.025,
             label="Predicted probability P(Adelie)")
plt.tight_layout()
plt.show()

# Independent consistency checks: both definitions of boundary mark score ~= 0, p ~= 0.5.
assert np.allclose(probability_surface, sigmoid(score_surface))
assert np.array_equal(binary_pipeline.predict(mesh_df),
                      (score_surface.ravel() > 0).astype(int))

**Interpretation.** The dark curve is actually a **straight line** in the original two-measurement plane; it marks the model's **$z=0$, or $p=0.5$, boundary**. Panel B shades the *continuous* probability field: probabilities vary smoothly even though the predicted class flips at the line. Training observations were used for fitting; the **starred** validation rows are overlaid **only to inspect unseen examples**. All plotted coordinates are in **millimeters**, while standardization happens **inside the fitted pipeline**.

Measurements overlap, so observations close to the learned boundary often receive probabilities near 0.5. That indicates indecision **within this model**, not a guarantee that every output is perfectly calibrated. A few individual predictions are not sufficient to judge model quality; L14 will address that question.

### One prediction, traced through every mathematical step

To connect the API back to the equations, take one held-out penguin. We can inspect its raw inputs, transformed features, linear score, sigmoid probability, and predicted class. Importantly, **we do not refit any preprocessing step on this observation**.

In [ ]:
row_id = 335
raw_one = X_val.loc[[row_id]]
one_imputed = binary_pipeline.named_steps["imputer"].transform(raw_one)
one_scaled = binary_pipeline.named_steps["scaler"].transform(one_imputed)
manual_z = learned_b0 + float(one_scaled[0] @ learned_b)
manual_p = float(sigmoid(manual_z))
api_p = float(binary_pipeline.predict_proba(raw_one)[0, positive_col])
api_label = int(binary_pipeline.predict(raw_one)[0])

print("Original penguin row:", row_id, "| actual species:", validation.loc[row_id, "species"])
display(raw_one)
print("Standardized measurements:", np.round(one_scaled[0], 3))
print(f"Score z = {manual_z:.4f}")
print(f"Sigmoid(z) = {manual_p:.4f} | API P(Adelie) = {api_p:.4f}")
print("Predicted class:", "Adelie" if api_label == 1 else "Not Adelie")
assert np.isclose(manual_p, api_p, atol=1e-12)

**Interpretation.** This one-row calculation shows that the pipeline does not hide a different idea: it performs the **same linear-score and sigmoid computation** we studied by hand, using coefficients and scaling parameters learned on training data.

For a genuinely new penguin, we would supply its measured features without the actual species. A predicted label is **an output to investigate**, not proof that the model has identified the species reliably.

In [ ]:
# Illustrative NEW penguin whose true species is unknown.
new_penguin = pd.DataFrame({"bill_length_mm": [42.0], "bill_depth_mm": [18.0]})
new_probability = float(binary_pipeline.predict_proba(new_penguin)[0, positive_col])
new_label = int(binary_pipeline.predict(new_penguin)[0])
print(f"For bill length = 42.0 mm and bill depth = 18.0 mm:")
print(f"  P(Adelie) = {new_probability:.3f}")
print("  Predicted class =", "Adelie" if new_label else "Not Adelie")

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:17px 21px;border-radius:8px;margin-top:28px;margin-bottom:13px;">
<div style="color:#67DED3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08</div>
<div style="font-size:24px;font-weight:700;line-height:1.4;">Three species: multiclass logistic regression and softmax</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Return to the original scientific question without turning L13 into an evaluation lecture.</div>
</div>

### Binary was a learning simplification. The original target has three species.

Our first question was: *Adelie or not Adelie?* Now return to the original goal:

> **Is this penguin Adelie, Chinstrap, or Gentoo?**

In **multiclass classification**, the classes are mutually exclusive and the model chooses **one** of three or more labels. A single binary sigmoid **cannot directly produce three class probabilities**.

A **multinomial logistic-regression model** instead learns a **linear score for each class**. For class $k$,

$$
z_k=b_{0,k}+b_{1,k}x_1+\cdots+b_{p,k}x_p.
$$

To turn $K$ scores into a probability distribution, apply **softmax**:

$$
\boxed{P(Y=k\mid X=x)=\frac{e^{z_k}}{\displaystyle\sum_{j=1}^{K} e^{z_j}}}.
$$

Here, $K$ is the number of species (3), $z_k$ is the score for species $k$, and the denominator adds the exponentials of **all** class scores. Every output is positive, and the probabilities sum to **1**. The class with the **largest probability** becomes the predicted label.

### Worked softmax example — calculate before calling scikit-learn

Suppose a **constructed** three-class model produces:

| Species | Score $z_k$ | $e^{z_k}$ (approx.) |
|:--|--:|--:|
| Adelie | 1 | 2.718 |
| Chinstrap | 0 | 1.000 |
| Gentoo | −1 | 0.368 |

The denominator is $2.718+1+0.368\approx4.086$. Therefore,

$$
\begin{aligned}
P(\text{Adelie})&\approx2.718/4.086=0.665,\\
P(\text{Chinstrap})&\approx1/4.086=0.245,\\
P(\text{Gentoo})&\approx0.368/4.086=0.090.
\end{aligned}
$$

The probabilities total approximately 1, and **Adelie** is the predicted class because **0.665** is largest. Notice: in multiclass prediction, **the winning probability need not exceed 0.5**.

In [ ]:
# Verify the hand-worked softmax with a numerically stable implementation.
def softmax(scores):
    """Convert an array of class scores to probabilities along its last axis."""
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / weights.sum(axis=-1, keepdims=True)

manual_scores = np.array([1.0, 0.0, -1.0])
manual_softmax = softmax(manual_scores)
manual_multiclass = pd.DataFrame({"species": SPECIES_ORDER,
                                  "score z": manual_scores,
                                  "exp(z)": np.exp(manual_scores),
                                  "softmax probability": manual_softmax})
display(manual_multiclass.round(3))
print("Sum of probabilities:", manual_softmax.sum())
print("Predicted class:", SPECIES_ORDER[np.argmax(manual_softmax)])
assert np.allclose(manual_softmax, [0.66524096, 0.24472847, 0.09003057], atol=1e-8)
assert np.isclose(manual_softmax.sum(), 1.0)

fig, ax = plt.subplots(figsize=(7.3, 3.6))
bars = ax.bar(SPECIES_ORDER, manual_softmax,
              color=[SPECIES_COLORS[s] for s in SPECIES_ORDER], width=.55)
for bar, value in zip(bars, manual_softmax):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+.025,
            f"{value:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.set(ylabel="Softmax probability", ylim=(0, 1),
       title="Illustrative three-class probability distribution")
ax.grid(axis="y", alpha=.15)
plt.show()

**Interpretation.** The three scores become three **coupled** probabilities: increasing one class score relative to the others changes their shares of the total. The subtraction of the largest score inside the Python function is a **numerical-stability technique**; subtracting the same constant from every score leaves the softmax probabilities unchanged. These numbers are illustrative, not fitted Penguins results.

A useful connection: the sigmoid is the **two-class softmax** when the two class scores are represented as $0$ and $z$. 

### Fit a real three-species model

We keep the **same 239 training rows and 68 validation rows**, but switch the target back to the **original species labels** and use four measured features.

In [ ]:
# Four physical features, original three-species target, same classroom split.
MULTICLASS_FEATURES = MEASUREMENTS.copy()
X_multi_train = train[MULTICLASS_FEATURES].copy()
y_multi_train = train["species"].copy()
X_multi_val = validation[MULTICLASS_FEATURES].copy()

multiclass_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))  # Multinomial for >=3 classes.
])
multiclass_pipeline.fit(X_multi_train, y_multi_train)
multiclass_model = multiclass_pipeline.named_steps["classifier"]
multi_classes = multiclass_model.classes_
multi_proba = multiclass_pipeline.predict_proba(X_multi_val)
multi_pred = multiclass_pipeline.predict(X_multi_val)

print("Probability column order from classes_:", multi_classes.tolist())
assert set(multi_classes) == set(SPECIES_ORDER)
assert multi_proba.shape == (len(X_multi_val), 3)
assert np.all((multi_proba >= 0) & (multi_proba <= 1))
assert np.allclose(multi_proba.sum(axis=1), 1.0)
assert np.array_equal(multi_pred, multi_classes[np.argmax(multi_proba, axis=1)])

multi_ids = [63, 245, 335, 330, 129]
multi_positions = X_multi_val.index.get_indexer(multi_ids)
multi_table = pd.DataFrame(multi_proba[multi_positions],
                           columns=[f"P({name})" for name in multi_classes],
                           index=multi_ids)
multi_table.insert(0, "Actual species", validation.loc[multi_ids, "species"])
multi_table["Predicted species"] = multi_pred[multi_positions]
multi_table["Probability sum"] = multi_proba[multi_positions].sum(axis=1)
display(multi_table.round(3))

**Interpretation.** The three probability columns are labeled using the fitted estimator's **`classes_`** order, not guessed. Each row sums to approximately one; `predict()` selects the highest-probability species. For example, row **129** produces probabilities close to **0.405 Adelie**, **0.401 Chinstrap**, and **0.194 Gentoo**. The predicted label is Adelie despite the winning probability being below 0.5.

That example is a reminder that **probabilities and hard labels carry different information**. We should never assume the maximum-probability class is necessarily a highly confident or correct prediction.

In [ ]:
# Extra conceptual check: scikit-learn's fitted multinomial scores use SOFTMAX.
one_multiclass_id = 129
one_multi_x = X_multi_val.loc[[one_multiclass_id]]
learned_class_scores = multiclass_pipeline.decision_function(one_multi_x)[0]
prob_from_scores = softmax(learned_class_scores)
prob_from_api = multiclass_pipeline.predict_proba(one_multi_x)[0]

display(pd.DataFrame({"species": multi_classes,
                      "learned score": learned_class_scores,
                      "softmax(score)": prob_from_scores,
                      "predict_proba": prob_from_api}).round(4))
assert np.allclose(prob_from_scores, prob_from_api, atol=1e-12)
print("Check: softmax(fitted class scores) matches predict_proba() for all classes.")

**Interpretation.** This checks the **actual multiclass mathematics**, not merely a manually invented example: scikit-learn's multinomial class scores transform through softmax into the probability vector returned by `predict_proba()`. We do **not** derive multinomial log loss, its gradients, or optimizer updates in this lecture. Those connections return in L26 and the neural-network sequence.

### Today's complete conceptual picture

| Question | Binary logistic regression | Multiclass logistic regression |
|:--|:--|:--|
| What comes from the features? | One linear score $z$ | One linear score $z_k$ per class |
| How do scores become probabilities? | **Sigmoid** | **Softmax** |
| How many probabilities? | Two: $p$ and $1-p$ | One per class; all sum to 1 |
| How do we choose a label? | A stated binary threshold | Largest class probability |
| How does the model learn? | Penalize probabilities inconsistent with the observed labels | Same guiding principle; details postponed |

**Closing question for L14:** We can now generate binary and multiclass predictions for penguins we did not train on. **How do we determine whether these classifications are reliable?** A handful of examples cannot answer that. In L14, we will study evaluation systematically—including disagreements between true and predicted labels, class-specific performance, and when decision rules matter.


### Acknowledgments, data provenance, and technical references

- **Palmer Penguins:** Original observations collected by Dr. Kristen Gorman and the Palmer Station Long Term Ecological Research program. Horst, A. M., Hill, A. P., and Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*, [doi:10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218). Data are distributed under CC0. 
- **Prior course lectures:** L07–L10 for Penguins data-quality decisions, feature representations, and the classroom split; L11–L12 for linear equations, model fitting, and train/validation/test discipline.
- **Pedagogical inspiration:** Prof. Antonio Liotta's materials on supervised classification, holdout validation, and linear separating boundaries. All plots and logistic-regression explanations in this notebook are original.
- **scikit-learn documentation:** [LogisticRegression](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html), [linear-model guide](https://scikit-learn.org/1.8/modules/linear_model.html#logistic-regression), [Pipeline](https://scikit-learn.org/1.8/modules/generated/sklearn.pipeline.Pipeline.html), [StandardScaler](https://scikit-learn.org/1.8/modules/generated/sklearn.preprocessing.StandardScaler.html).
- **Software:** Python 3.11+, NumPy, pandas, Matplotlib, scikit-learn, Jupyter Notebook or JupyterLab. No network access or separate datasets are needed to run any code cell.